In [11]:
import torch
import re
import pandas as pd
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

In [9]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [10]:
train_data 

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."
...,...,...,...
14727,13863028,Romeo: You are on my ‘People you may know’ lis...,Romeo is trying to get Greta to add him to her...
14728,13828570,Theresa: <file_photo>\r\nTheresa: <file_photo>...,Theresa is at work. She gets free food and fre...
14729,13819050,John: Every day some bad news. Japan will hunt...,Japan is going to hunt whales again. Island an...
14730,13828395,Jennifer: Dear Celia! How are you doing?\r\nJe...,Celia couldn't make it to the afternoon with t...


In [15]:
def data_clean(text):
    text = re.sub(r"\r\n", " ", text) # removes lines
    text = re.sub(r"\s+", " ", text) # Extra spaces
    text = text.strip().lower() # remove leading and trailing space and lowercase

    return text


In [18]:
train = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [21]:
train["dialogue"] = train["dialogue"].apply(data_clean)
train["summary"] = train["summary"].apply(data_clean)

val["dialogue"] = val["dialogue"].apply(data_clean)
val["summary"] = val["summary"].apply(data_clean)

In [22]:
tokenizer =  T5Tokenizer.from_pretrained("t5-small")

In [24]:
def fine_tuning(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length= 512, truncation=True)
    target = tokenizer(data["summary"], padding="max_length", max_length= 150, truncation=True)

    inputs["labels"] = target["input_ids"] # Token ids add to inputs as labels

    return inputs

In [29]:
train_dataset = train.apply(fine_tuning, axis=1).to_list()
val_dataset = val.apply(fine_tuning, axis=1).to_list()

In [25]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device: ", device)

Device:  cuda


In [26]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")
model.to(device)

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 10394.51it/s]


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [30]:
train_args = TrainingArguments(output_dir="./results",
                              per_device_train_batch_size= 8,
                              per_device_eval_batch_size= 8,
                              weight_decay= 0.005,
                              num_train_epochs= 6,
                              warmup_steps=500,
                              eval_strategy="epoch",
                              save_strategy="epoch"
)

In [32]:
trainer = Trainer(model=model,
                  args = train_args,
                  train_dataset=train_dataset,
                  eval_dataset=val_dataset
                  )

In [33]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.610949,0.380379
2,0.395872,0.359301
3,0.373088,0.353730
4,0.360983,0.349595
5,0.354600,0.349208
6,0.350778,0.348555


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.53it/s]


TrainOutput(global_step=3000, training_loss=0.9077117513020834, metrics={'train_runtime': 1052.8451, 'train_samples_per_second': 22.795, 'train_steps_per_second': 2.849, 'total_flos': 3248203235328000.0, 'train_loss': 0.9077117513020834, 'epoch': 6.0})

In [34]:
model.save_pretrained("./saved_summary")
tokenizer.save_pretrained("./saved_summary")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.24it/s]


('./saved_summary/tokenizer_config.json', './saved_summary/tokenizer.json')

In [35]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary")
tokenizers = T5Tokenizer.from_pretrained("./saved_summary")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 6546.26it/s]


In [38]:
def summarization_text(dialogue):
    dialogue = data_clean(dialogue)
    inputs = tokenizer(dialogue,
                       padding="max_length",
                       max_length = 512,
                       truncation= True,
                       return_tensors = "pt").to(device)

    model.to(device)
    model.eval()

    targets = model.generate(input_ids =inputs["input_ids"],
                            attention_mask = inputs["attention_mask"],
                            max_length=150,
                            min_length=10,
                            num_beams=4,
                            early_stopping=True)

    summary = tokenizer.decode(targets[0], 
                               skip_special_tokens=True)


    return summary

    

    

In [39]:


dialogue = """
John: Hi Sarah, are you coming to the meeting today?
Sarah: Yes, I'll be there in 15 minutes.
John: Great! Don't forget to bring the project report.
Sarah: I have it with me.
John: Perfect. See you soon.
Sarah: See you.
"""

summary = summarization_text(dialogue)

print("Summary: ", summary)



Summary:  sarah will be at the meeting today in 15 minutes. john will bring the project report with her.
